# 📊 M5 Walmart Forecasting — EDA & Model Performance Analysis
Deep dive into correlations, seasonality, event effects, and model diagnostics.


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy.stats import linregress
import warnings
warnings.filterwarnings('ignore')

# Aesthetic config
plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor':   '#1a1d27',
    'axes.edgecolor':   '#3a3d4a',
    'axes.labelcolor':  '#c8ccd8',
    'xtick.color':      '#c8ccd8',
    'ytick.color':      '#c8ccd8',
    'text.color':       '#e8ecf4',
    'grid.color':       '#2a2d3a',
    'grid.alpha':       0.5,
    'font.family':      'monospace',
    'axes.titlesize':   13,
    'axes.titleweight': 'bold',
})
PALETTE = ['#4fc3f7','#81c784','#ffb74d','#e57373','#ce93d8','#4db6ac','#f06292','#aed581','#90caf9','#fff176']

train = pd.read_csv('data/train.csv')
cal   = pd.read_csv('data/calendar_events.csv')

train.columns = train.columns.str.strip().str.lower()
cal.columns   = cal.columns.str.strip().str.lower()

train['date'] = pd.to_datetime(train['date'])
cal['date']   = pd.to_datetime(cal['date'])

stores = train[train['store_id'] > 0].copy()
stores['dow']   = stores['date'].dt.dayofweek
stores['month'] = stores['date'].dt.month
stores['year']  = stores['date'].dt.year
stores['is_weekend'] = stores['dow'].isin([5,6]).astype(int)

store_names = stores.drop_duplicates('store_id').set_index('store_id')['store_name'].to_dict()

print("Loaded:", stores.shape)
stores.head(3)


## 1. Revenue Over Time — Per Store

In [ ]:

fig, axes = plt.subplots(5, 2, figsize=(18, 20), facecolor='#0f1117')
fig.suptitle('Daily Revenue — All Stores (2011–2015)', fontsize=16, color='#e8ecf4', y=0.98)

for ax, (sid, grp) in zip(axes.flat, stores.groupby('store_id')):
    grp = grp.sort_values('date')
    ax.plot(grp['date'], grp['revenue'], alpha=0.4, linewidth=0.6, color=PALETTE[sid % len(PALETTE)])
    # 28-day rolling mean
    rolled = grp.set_index('date')['revenue'].rolling(28).mean()
    ax.plot(rolled.index, rolled.values, linewidth=2, color=PALETTE[sid % len(PALETTE)])
    ax.set_title(store_names[sid], fontsize=10)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=30, labelsize=7)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 2. Day-of-Week × Store Revenue Heatmap

In [ ]:

dow_pivot = stores.groupby(['store_id','dow'])['revenue'].mean().unstack()
dow_pivot.index = [store_names[i].split('–')[-1].strip() for i in dow_pivot.index]
dow_pivot.columns = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']

fig, ax = plt.subplots(figsize=(12, 6), facecolor='#0f1117')
sns.heatmap(dow_pivot, annot=True, fmt='.0f', cmap='YlOrRd',
            linewidths=0.5, linecolor='#0f1117', ax=ax, cbar_kws={'shrink':0.7})
ax.set_title('Average Revenue by Store × Day of Week', fontsize=14, pad=15)
ax.set_ylabel('')
plt.tight_layout()
plt.show()


## 3. Weekend Premium by Store

In [ ]:

wk = stores.groupby(['store_id','is_weekend'])['revenue'].mean().unstack()
wk.columns = ['Weekday','Weekend']
wk['Premium %'] = (wk['Weekend']/wk['Weekday']-1)*100
wk['store'] = [store_names[i].split('–')[-1].strip() for i in wk.index]
wk = wk.sort_values('Premium %', ascending=True)

fig, ax = plt.subplots(figsize=(12, 5), facecolor='#0f1117')
bars = ax.barh(wk['store'], wk['Premium %'], color=PALETTE[:len(wk)], edgecolor='none')
for bar, val in zip(bars, wk['Premium %']):
    ax.text(val + 0.3, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=10, color='#e8ecf4')
ax.set_title('Weekend Revenue Premium (%) by Store', fontsize=14)
ax.set_xlabel('% above weekday average')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()


## 4. Cross-Store Revenue Correlations

In [ ]:

pivot = stores.pivot(index='date', columns='store_id', values='revenue')
corr  = pivot.corr()
labels = [store_names[i].split('–')[-1].strip() for i in corr.index]

fig, ax = plt.subplots(figsize=(11, 9), facecolor='#0f1117')
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, vmin=0.2, vmax=1.0,
            xticklabels=labels, yticklabels=labels,
            linewidths=0.4, ax=ax, cbar_kws={'shrink':0.7})
ax.set_title('Cross-Store Revenue Correlation Matrix', fontsize=14, pad=15)
plt.tight_layout()
plt.show()


## 5. Year-over-Year Growth Trend

In [ ]:

yoy = stores.groupby(['store_id','year'])['revenue'].mean().unstack()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6), facecolor='#0f1117')

for i, sid in enumerate(yoy.index):
    ax1.plot(yoy.columns, yoy.loc[sid], marker='o', linewidth=2,
             color=PALETTE[i], label=store_names[sid].split('–')[-1].strip())
ax1.set_title('Annual Average Revenue per Store', fontsize=13)
ax1.set_xlabel('Year')
ax1.set_ylabel('Avg Daily Revenue ($)')
ax1.legend(fontsize=8, loc='upper left')
ax1.grid(True, alpha=0.3)

# Growth rate
growth = ((yoy[2015] - yoy[2011]) / yoy[2011] * 100).sort_values(ascending=True)
labels = [store_names[i].split('–')[-1].strip() for i in growth.index]
bars = ax2.barh(labels, growth.values, color=PALETTE[:len(growth)], edgecolor='none')
for bar, val in zip(bars, growth.values):
    ax2.text(val + 0.5, bar.get_y() + bar.get_height()/2,
             f'{val:.0f}%', va='center', fontsize=9)
ax2.set_title('Total Revenue Growth 2011→2015', fontsize=13)
ax2.set_xlabel('Growth (%)')
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()


## 6. Autocorrelation Structure (Lag Analysis)

In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(20, 8), facecolor='#0f1117')
fig.suptitle('Autocorrelation by Lag — Each Store', fontsize=14, color='#e8ecf4')

lags = list(range(1, 57))

for ax, (sid, grp) in zip(axes.flat, stores.groupby('store_id')):
    s = grp.sort_values('date')['revenue']
    acfs = [s.autocorr(lag=l) for l in lags]
    colors = ['#4fc3f7' if v > 0 else '#e57373' for v in acfs]
    ax.bar(lags, acfs, color=colors, edgecolor='none', width=0.8)
    ax.axhline(0, color='white', linewidth=0.5)
    ax.axvline(7,  color='#ffb74d', linewidth=1, linestyle='--', alpha=0.7, label='lag-7')
    ax.axvline(28, color='#81c784', linewidth=1, linestyle='--', alpha=0.7, label='lag-28')
    ax.set_title(store_names[sid].split('–')[-1].strip(), fontsize=9)
    ax.set_ylim(-0.3, 1.0)
    ax.set_xlabel('Lag (days)', fontsize=8)
    ax.grid(True, alpha=0.2)

axes.flat[0].legend(fontsize=7)
plt.tight_layout()
plt.show()


## 7. Revenue Lift Around Events (−7 to +7 days)

In [ ]:

cal_dates = set(cal['date'])
baseline_dow = stores.groupby(['store_id','dow'])['revenue'].mean()

offsets = range(-7, 8)
lifts = []
for offset in offsets:
    shifted_dates = {d + pd.Timedelta(days=offset) for d in cal_dates}
    tmp = stores[stores['date'].isin(shifted_dates)].copy()
    if len(tmp):
        bl = baseline_dow[pd.MultiIndex.from_arrays([tmp['store_id'], tmp['dow']])].values
        lift = (tmp['revenue'].values / bl - 1).mean() * 100
        lifts.append({'offset': offset, 'lift': lift})

df_lift = pd.DataFrame(lifts)

fig, ax = plt.subplots(figsize=(13, 5), facecolor='#0f1117')
colors = ['#81c784' if v > 0 else '#e57373' for v in df_lift['lift']]
ax.bar(df_lift['offset'], df_lift['lift'], color=colors, edgecolor='none', width=0.8)
ax.axhline(0, color='white', linewidth=0.8)
ax.axvline(0, color='#ffb74d', linewidth=2, linestyle='--', label='Event day', alpha=0.9)
ax.set_title('Average Revenue Lift (%) by Days Relative to Any Event', fontsize=14)
ax.set_xlabel('Days from Event (negative = before event)')
ax.set_ylabel('Avg Lift vs Same-DoW Baseline (%)')
ax.legend()
ax.grid(True, alpha=0.3)
for _, row in df_lift.iterrows():
    ax.text(row['offset'], row['lift'] + (0.1 if row['lift'] >= 0 else -0.2),
            f"{row['lift']:.1f}%", ha='center', fontsize=8, color='#e8ecf4')
plt.tight_layout()
plt.show()


## 8. Revenue Lift by Event Type

In [ ]:

stores2 = stores.merge(cal, on='date', how='left')
baseline = stores2.groupby(['store_id','dow'])['revenue'].transform('mean')
stores2['lift'] = (stores2['revenue'] - baseline) / baseline * 100
event_lift = stores2[stores2['event'].notna()].groupby('event')['lift'].mean().sort_values()

fig, ax = plt.subplots(figsize=(14, 9), facecolor='#0f1117')
colors = ['#81c784' if v > 0 else '#e57373' for v in event_lift.values]
ax.barh(event_lift.index, event_lift.values, color=colors, edgecolor='none')
ax.axvline(0, color='white', linewidth=0.8)
for i, (name, val) in enumerate(event_lift.items()):
    ax.text(val + (0.2 if val >= 0 else -0.2), i,
            f'{val:.1f}%', va='center', ha='left' if val >= 0 else 'right', fontsize=8)
ax.set_title('Average Revenue Lift (%) by Event Type', fontsize=14)
ax.set_xlabel('Lift vs Same-DoW Baseline (%)')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()


## 9. Monthly Seasonality Pattern

In [ ]:

month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig, axes = plt.subplots(2, 5, figsize=(20, 8), facecolor='#0f1117')
fig.suptitle('Monthly Revenue Pattern — Per Store', fontsize=14, color='#e8ecf4')

for ax, (sid, grp) in zip(axes.flat, stores.groupby('store_id')):
    monthly = grp.groupby('month')['revenue'].mean()
    ax.bar(monthly.index, monthly.values, color=PALETTE[sid % len(PALETTE)], edgecolor='none')
    ax.set_xticks(range(1,13))
    ax.set_xticklabels(month_names, rotation=45, fontsize=7)
    ax.set_title(store_names[sid].split('–')[-1].strip(), fontsize=9)
    ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()


## 10. Model Performance Diagnostics

In [ ]:

import os
if not os.path.exists('data/valid_predictions.csv'):
    print("Run improved_model.py first to generate validation predictions.")
else:
    vdf = pd.read_csv('data/valid_predictions.csv')
    vdf['date'] = pd.to_datetime(vdf['date'])
    vdf['residual'] = vdf['revenue'] - vdf['pred']
    vdf['abs_err'] = vdf['residual'].abs()
    vdf['pct_err']  = (vdf['residual'] / vdf['revenue'].clip(1)) * 100
    from sklearn.metrics import mean_squared_error
    rmse = np.sqrt(mean_squared_error(vdf['revenue'], vdf['pred']))
    mae  = vdf['abs_err'].mean()
    mape = vdf['pct_err'].abs().mean()
    print(f"Validation RMSE : {rmse:,.2f}")
    print(f"Validation MAE  : {mae:,.2f}")
    print(f"Validation MAPE : {mape:.2f}%")


In [ ]:

if os.path.exists('data/valid_predictions.csv'):
    fig, axes = plt.subplots(2, 3, figsize=(20, 12), facecolor='#0f1117')
    fig.suptitle('Model Performance Diagnostics', fontsize=15, color='#e8ecf4')

    # 1. Actual vs Predicted scatter
    ax = axes[0,0]
    ax.scatter(vdf['revenue'], vdf['pred'], alpha=0.15, s=8, color='#4fc3f7')
    mn, mx = vdf['revenue'].min(), vdf['revenue'].max()
    ax.plot([mn, mx], [mn, mx], 'r--', linewidth=1.5, label='Perfect')
    ax.set_xlabel('Actual Revenue')
    ax.set_ylabel('Predicted Revenue')
    ax.set_title('Actual vs Predicted')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    # 2. Residuals over time
    ax = axes[0,1]
    for i, (sid, grp) in enumerate(vdf.groupby('store_id')):
        grp = grp.sort_values('date')
        ax.plot(grp['date'], grp['residual'], alpha=0.6, linewidth=0.8,
                color=PALETTE[i], label=store_names.get(sid,''))
    ax.axhline(0, color='white', linewidth=1)
    ax.set_title('Residuals Over Time')
    ax.set_xlabel('Date')
    ax.set_ylabel('Actual − Predicted')
    ax.grid(True, alpha=0.3)

    # 3. Residual distribution
    ax = axes[0,2]
    ax.hist(vdf['residual'], bins=80, color='#ce93d8', edgecolor='none', alpha=0.85)
    ax.axvline(0, color='#ffb74d', linewidth=2)
    ax.axvline(vdf['residual'].mean(), color='#e57373', linewidth=1.5, linestyle='--',
               label=f"Mean: {vdf['residual'].mean():.0f}")
    ax.set_title('Residual Distribution')
    ax.set_xlabel('Residual ($)')
    ax.set_ylabel('Count')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    # 4. RMSE by store
    ax = axes[1,0]
    rmse_store = vdf.groupby('store_id').apply(
        lambda g: np.sqrt(mean_squared_error(g['revenue'], g['pred']))).sort_values(ascending=True)
    labels = [store_names.get(i,'').split('–')[-1].strip() for i in rmse_store.index]
    bars = ax.barh(labels, rmse_store.values, color=PALETTE[:len(rmse_store)], edgecolor='none')
    for bar, val in zip(bars, rmse_store.values):
        ax.text(val+5, bar.get_y()+bar.get_height()/2, f'{val:,.0f}', va='center', fontsize=8)
    ax.set_title('RMSE by Store')
    ax.set_xlabel('RMSE ($)')
    ax.grid(axis='x', alpha=0.3)

    # 5. % Error by day of week
    ax = axes[1,1]
    dow_err = vdf.groupby('dow')['pct_err'].apply(lambda x: x.abs().mean())
    ax.bar(['Mon','Tue','Wed','Thu','Fri','Sat','Sun'], dow_err.values,
           color=['#e57373' if d in [5,6] else '#4fc3f7' for d in range(7)], edgecolor='none')
    ax.set_title('Mean Absolute % Error by Day of Week')
    ax.set_ylabel('MAPE (%)')
    ax.grid(axis='y', alpha=0.3)

    # 6. Feature importance
    ax = axes[1,2]
    if os.path.exists('data/feature_importance.csv'):
        fi = pd.read_csv('data/feature_importance.csv', index_col=0).squeeze()
        top = fi.sort_values(ascending=True).tail(20)
        ax.barh(top.index, top.values, color='#81c784', edgecolor='none')
        ax.set_title('Top 20 Feature Importances')
        ax.set_xlabel('Importance (gain)')
        ax.grid(axis='x', alpha=0.3)

    plt.tight_layout()
    plt.show()


## 11. Model Error Near Events

In [ ]:

if os.path.exists('data/valid_predictions.csv'):
    vdf['days_from_event'] = vdf['date'].apply(
        lambda d: min(abs((d - e).days) for e in cal['date']) )
    bins = [0,1,2,3,7,14,100]
    labels_b = ['Event day','1 day','2 days','3-7 days','1-2 weeks','2+ weeks']
    vdf['proximity_bin'] = pd.cut(vdf['days_from_event'], bins=bins, labels=labels_b, right=False)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6), facecolor='#0f1117')
    
    rmse_prox = vdf.groupby('proximity_bin', observed=True).apply(
        lambda g: np.sqrt(mean_squared_error(g['revenue'], g['pred'])))
    ax1.bar(rmse_prox.index, rmse_prox.values, color='#ffb74d', edgecolor='none')
    ax1.set_title('RMSE by Distance to Nearest Event')
    ax1.set_xlabel('Distance to event')
    ax1.set_ylabel('RMSE ($)')
    ax1.tick_params(axis='x', rotation=25)
    ax1.grid(axis='y', alpha=0.3)

    mape_prox = vdf.groupby('proximity_bin', observed=True)['pct_err'].apply(lambda x: x.abs().mean())
    ax2.bar(mape_prox.index, mape_prox.values, color='#f06292', edgecolor='none')
    ax2.set_title('MAPE by Distance to Nearest Event')
    ax2.set_xlabel('Distance to event')
    ax2.set_ylabel('MAPE (%)')
    ax2.tick_params(axis='x', rotation=25)
    ax2.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.show()


## 12. Actual vs Predicted — Per Store (Validation Period)

In [ ]:

if os.path.exists('data/valid_predictions.csv'):
    fig, axes = plt.subplots(5, 2, figsize=(18, 20), facecolor='#0f1117')
    fig.suptitle('Actual vs Predicted — Validation Period', fontsize=15, color='#e8ecf4')

    for ax, (sid, grp) in zip(axes.flat, vdf.groupby('store_id')):
        grp = grp.sort_values('date')
        ax.plot(grp['date'], grp['revenue'], label='Actual', color='#4fc3f7', linewidth=1.5)
        ax.plot(grp['date'], grp['pred'],    label='Predicted', color='#ffb74d',
                linewidth=1.5, linestyle='--')
        ax.fill_between(grp['date'], grp['revenue'], grp['pred'],
                        alpha=0.15, color='#e57373')
        ax.set_title(store_names.get(sid,''), fontsize=9)
        ax.legend(fontsize=7)
        ax.grid(True, alpha=0.3)
        ax.tick_params(axis='x', rotation=30, labelsize=7)

    plt.tight_layout()
    plt.show()
